In [ ]:
# load cleaned data
import pickle 
with open("transcripts_clean.pickle", "rb") as f:
    transcripts = pickle.load(f)

In [ ]:
import polars as pl
import re


In [ ]:
# the dictionary we had before, was a bit complicated to work with, so let's just make a big df out of all the transcripts
df_all = pl.concat([
    df.with_columns(pl.lit(name).alias("session"))
    for name, df in transcripts.items()
], how="vertical_relaxed")


In [ ]:
df_all["session"].unique()



In [ ]:
# Let's normalize the session names, so they're shorter and easier to handle
df_all = df_all.with_columns(
    pl.col("session")
        .str.extract(r"(\d+)") # some regex to extract only the digits (1 or 2)
        .alias("session")
).with_columns(
    pl.when(pl.col("session").str.len_chars()<2) # check for single digits
        .then(pl.lit("0") + pl.col("session")) # add a zero there, so that the numbers are uniform -> 01 02 etc
        .otherwise(pl.col("session"))
        .alias("session")
)

df_all["session"].unique() # now this is nice and clean 


In [ ]:
# some funny first analysis! Let's find out in which session there was most laughter
# in the transcripts they are coded as *laughing* *laughing under breath* *LAUGHS* etc. 
funny_sessions = df_all.with_columns(
    pl.col("speech")
      .str.count_matches(r"(?i)\*[^*]*laugh[^*]*\*") # for every line in speech count anytime there is a string inbetween two * that contains "laugh" (lower or upper case)
      .alias("laugh_count")
).group_by("session").agg( # do for every session
    pl.col("laugh_count").sum() 
).sort("session")

funny_sessions.sort("laugh_count", descending=True) # apparently session 5 was the funiest... that was the pandora's box text, we have read together as well, what do we think about that?

In [ ]:
# Remember our example question from session two? 

turns = df_all.with_columns(
    (pl.col("speaker") != pl.col("speaker").shift(1)) # compare current to next speaker (shift(1) looks in next row, -1 previous)
    .over("session")   # do the comparison within each session separately
    .fill_null(False)  # first row of each session is not a shift
    .alias("is_shift")
).group_by("session", "speaker").agg(
    pl.col("is_shift").sum().alias("speaker_shifts")
).filter(
    pl.col("speaker").is_in(["NA", "GROUP", "Researcher"]).not_() # those don't really interest us at this point
    ).sort("session", "speaker_shifts", descending=True)

turns

In [ ]:
import altair as alt
# let's plot this!

turn_chart = alt.Chart(turns).mark_bar().encode( # we are plotting our df turns, to a bar-plot
    x=alt.X("session:N"), # session on x axis (N = Nominal, we don't want to read it as number)
    y=alt.Y("speaker_shifts:Q"), # (Q = quantitative - this is a number)
    color="speaker:N", # colourful speakers!
    xOffset="speaker:N" # bars next to each other
)

turn_chart

In [ ]:
df_stars = df_all.with_columns(
    pl.col("speech")
    .str.count_matches(r"\*")
    .alias("countstars")
)

odd_stars = df_stars.filter(
    pl.col("countstars") % 2 != 0
)
odd_stars

In [ ]:
df_stars

In [ ]:
df_cc = df_stars.with_columns(
    pl.col("speech")
    .str.replace("*IS.", "*IS*", literal=True)
    .str.replace("*OS**", "*OS*", literal=True)
    .str.replace("*IS ", "*IS* ", literal=True)
    .str.replace("*OSÜ", "*OS*", literal=True)
    .alias("speech")
).with_columns(
    pl.when(pl.col("countstars") == 1)
    .then(pl.col("speech").str.replace("*laughter", "*laughter*", literal=True))
    .otherwise(pl.col("speech"))
    .alias("speech")
).with_columns(
    pl.col("speech")
    .str.count_matches(r"\*")
    .alias("countstars")
).with_columns(
    pl.when(pl.col("countstars") == 1)
    .then(pl.col("speech").str.replace("*IS", "*IS*", literal=True))
    .otherwise(pl.col("speech"))
    .alias("speech")
).with_columns(
    pl.col("speech")
    .str.count_matches(r"\*")
    .alias("countstars")
).with_columns(
    pl.when(pl.col("countstars") == 1)
    .then(pl.col("speech").str.replace(r"\*", ""))
    .otherwise(pl.col("speech"))
    .alias("speech")
).with_columns(
    pl.col("speech")
    .str.count_matches(r"\*")
    .alias("countstars")
)
df_cc

In [ ]:
open_par = df_cc.with_columns(
    pl.col("speech")
    .str.count_matches(r"\(").alias("open"),
    pl.col("speech")
    .str.count_matches(r"\)").alias("close")
).filter((pl.col("open") + pl.col("close")) > 0)

open_par

In [ ]:
df_clean = df_cc.with_columns(
    pl.col("speech")
    .str.replace_all(r"\*[^*]*\*", "")
    .str.replace_all(r"\(\d+\.?\d*\)", "")
    .alias("speech")
)

In [ ]:
tokens = df_clean.with_columns(
    pl.col("speech")
    .str.extract_all(r"\b\w+\b").alias("speech")
).filter(pl.col("event").is_in(["Reading Story", "Reading Poem"]).not_() | pl.col("event").is_null()
).filter(pl.col("speaker").is_in(["NA", "GROUP", "Researcher"]).not_()
).with_columns(
    pl.col("speech")
    .list.len()
    .alias("n_words")
)
tokens

In [ ]:
speaker_tokens = tokens.group_by("session", "speaker").agg(
    pl.col("n_words").sum().alias("n_words")
).sort(["session", "n_words"], descending=True)
speaker_tokens


In [ ]:
# let's plot this!

token_chart = alt.Chart(
        speaker_tokens.filter(pl.col("n_words") > 0)
).mark_bar().encode( # we are plotting our df turns, to a bar-plot
    x=alt.X("session:N"), # session on x axis (N = Nominal, we don't want to read it as number)
    y=alt.Y("n_words:Q"), # (Q = quantitative - this is a number)
    color="speaker:N", # colourful speakers!
    xOffset="speaker:N" # bars next to each other
)
token_chart


In [ ]:
turn_chart

In [ ]:
import spacy 
spacy.cli.download('en_core_web_sm')
nlp = spacy.load('en_core_web_sm')

In [ ]:
spacy_pipeline.pipe_names

In [ ]:
session02 = df_clean.filter(pl.col("session") == "02").filter(pl.col("event").is_in(["Reading Story", "Reading Poem"]).not_() | pl.col("event").is_null())

speech02 = " ".join(session02["speech"].drop_nulls().to_list())
speech02

In [ ]:
doc = nlp(speech02)

In [ ]:
for token in doc[300:330]:
    print(token.text, token.lemma_, token.pos_, token.ent_type_)

In [ ]:
persons = []
for token in doc:
    if token.ent_type_ == "PERSON":
        persons.append(token.text)

print(set(persons))

In [ ]:
from spacy import displacy
sentences = list(doc.sents)
sentence = sentences[14]
displacy.render(sentence, style="dep", jupyter=True)

In [ ]:
displacy.render(doc, style="ent", jupyter=True)